# Chicago Crime - Notebook EDA

Ce notebook reprend la partie exploratoire du notebook `chicago_crime_ML_v3.ipynb` et la rend autonome.

Objectifs :
- charger les crimes de Chicago depuis un CSV local ou depuis l'API Chicago Data Portal ;
- nettoyer les champs utiles pour l'analyse ;
- comprendre les volumes, les types de crimes, les arrestations, la temporalite et la geographie ;
- produire des tableaux et graphiques directement reutilisables dans une presentation.


## Analyse du notebook existant

Le notebook original suit un pipeline complet :

1. Chargement des donnees Chicago depuis `https://data.cityofchicago.org/resource/ijzp-q8t2.csv` avec un filtre `year >= 2020`.
2. EDA rapide : volumes, valeurs manquantes, top crimes, taux d'arrestation par type et par heure.
3. Nettoyage et feature engineering : date, heure, jour, mois, weekend, domestic, groupes de lieux, categories de crimes.
4. K-Means geographique pour creer une feature `cluster`.
5. Random Forest pour predire le type de crime.
6. XGBoost pour predire l'arrestation en utilisant le type de crime predit.

Ce notebook garde les etapes 1 a 3 et enrichit l'EDA avec des vues temporelles, spatiales et des tableaux de synthese.


## 0. Imports et configuration


In [ ]:
from pathlib import Path
from io import StringIO
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

RANDOM_STATE = 42
START_YEAR = 2020
API_LIMIT = 200_000
DATA_URL = 'https://data.cityofchicago.org/resource/ijzp-q8t2.csv'

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
EDA_DIR = PROJECT_ROOT / 'artifacts' / 'eda'
DATA_DIR.mkdir(exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_CANDIDATES = [
    DATA_DIR / 'chicago_crimes.csv',
    DATA_DIR / 'chicago_crimes_sample.csv',
    PROJECT_ROOT / 'chicago_crimes.csv',
]

print(f'Project root : {PROJECT_ROOT}')
print(f'EDA outputs  : {EDA_DIR}')


## 1. Chargement des donnees


In [ ]:
def load_chicago_crimes(force_api=False):
    """Load Chicago crimes from a local CSV if available, otherwise from the public API."""
    if not force_api:
        for path in LOCAL_DATA_CANDIDATES:
            if path.exists():
                print(f'Chargement local : {path}')
                return pd.read_csv(path, low_memory=False)

    params = {
        '$limit': API_LIMIT,
        '$order': 'date DESC',
        '$where': f'year >= {START_YEAR}',
    }
    print('Chargement depuis l API Chicago...')
    response = requests.get(DATA_URL, params=params, timeout=120)
    response.raise_for_status()
    df_loaded = pd.read_csv(StringIO(response.text), low_memory=False)

    cache_path = DATA_DIR / 'chicago_crimes_sample.csv'
    df_loaded.to_csv(cache_path, index=False)
    print(f'Cache ecrit : {cache_path}')
    return df_loaded


df_raw = load_chicago_crimes()
print(f'Shape brute : {df_raw.shape}')
df_raw.head(3)


In [ ]:
print('Colonnes disponibles :')
print(df_raw.columns.tolist())

summary = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
    'nunique': df_raw.nunique(dropna=True),
}).sort_values('missing_pct', ascending=False)
summary.head(20)


## 2. Nettoyage et feature engineering


In [ ]:
def to_binary(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int)
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y']).astype(int)


def location_group(location):
    if pd.isna(location):
        return 'Other'
    loc = str(location).upper()
    if any(word in loc for word in ['STREET', 'ALLEY', 'SIDEWALK', 'PARKING']):
        return 'Street'
    if any(word in loc for word in ['RESIDENCE', 'APARTMENT', 'HOUSE']):
        return 'Residence'
    if any(word in loc for word in ['STORE', 'RESTAURANT', 'BANK', 'RETAIL', 'GAS']):
        return 'Commercial'
    if any(word in loc for word in ['CTA', 'BUS', 'TRAIN', 'VEHICLE']):
        return 'Transport'
    if any(word in loc for word in ['PARK', 'SCHOOL', 'CHURCH', 'HOSPITAL']):
        return 'Public'
    return 'Other'


def prepare_for_eda(df_input):
    df = df_input.copy()
    df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date_parsed', 'latitude', 'longitude', 'primary_type', 'arrest'])
    df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]

    df['year'] = df['date_parsed'].dt.year
    df['month'] = df['date_parsed'].dt.month
    df['month_name'] = df['date_parsed'].dt.month_name()
    df['hour'] = df['date_parsed'].dt.hour
    df['day_of_week'] = df['date_parsed'].dt.dayofweek
    df['day_name'] = df['date_parsed'].dt.day_name()
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    df['arrest'] = to_binary(df['arrest'])
    df['domestic'] = to_binary(df['domestic'])
    df['location_group'] = df['location_description'].apply(location_group)

    top10 = df['primary_type'].value_counts().head(10).index
    df['crime_cat'] = np.where(df['primary_type'].isin(top10), df['primary_type'], 'OTHER')
    return df


df = prepare_for_eda(df_raw)
print(f'Lignes brutes   : {len(df_raw):,}')
print(f'Lignes nettoyees: {len(df):,}')
print(f'Periode         : {df["date_parsed"].min().date()} -> {df["date_parsed"].max().date()}')
df.head(3)


## 3. Vue generale


In [ ]:
kpis = pd.Series({
    'lignes': len(df),
    'types_crimes': df['primary_type'].nunique(),
    'taux_arrestation': df['arrest'].mean(),
    'taux_domestic': df['domestic'].mean(),
    'part_weekend': df['is_weekend'].mean(),
})

display(kpis.to_frame('valeur'))

print(f"Taux d arrestation : {kpis['taux_arrestation']:.1%}")
print(f"Crimes domestiques : {kpis['taux_domestic']:.1%}")
print(f"Crimes le weekend  : {kpis['part_weekend']:.1%}")


In [ ]:
crime_summary = (
    df.groupby('crime_cat')
    .agg(
        volume=('crime_cat', 'size'),
        arrest_rate=('arrest', 'mean'),
        domestic_rate=('domestic', 'mean'),
        weekend_rate=('is_weekend', 'mean'),
    )
    .sort_values('volume', ascending=False)
)
crime_summary.style.format({
    'volume': '{:,.0f}',
    'arrest_rate': '{:.1%}',
    'domestic_rate': '{:.1%}',
    'weekend_rate': '{:.1%}',
})


## 4. Distribution des types de crimes


In [ ]:
top15 = df['primary_type'].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top15.index, top15.values, color='#4c78a8', edgecolor='white')
ax.set_title('Top 15 des types de crimes - Chicago', fontweight='bold')
ax.set_xlabel('Nombre de crimes')
for idx, value in enumerate(top15.values):
    ax.text(value, idx, f' {value:,}', va='center', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(EDA_DIR / '01_top_crimes.png', bbox_inches='tight')
plt.show()


## 5. Arrestations


In [ ]:
arrest_by_type = (
    df.groupby('crime_cat')['arrest']
    .agg(['mean', 'count'])
    .query('count >= 200')
    .sort_values('mean', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(arrest_by_type['mean'] >= df['arrest'].mean(), '#59a14f', '#e15759')
ax.barh(arrest_by_type.index, arrest_by_type['mean'] * 100, color=colors, edgecolor='white')
ax.axvline(df['arrest'].mean() * 100, color='black', linestyle='--', linewidth=1.4,
           label=f'Moyenne globale : {df["arrest"].mean() * 100:.1f}%')
ax.set_title('Taux d arrestation par categorie de crime', fontweight='bold')
ax.set_xlabel('Taux d arrestation (%)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(EDA_DIR / '02_arrest_rate_by_crime.png', bbox_inches='tight')
plt.show()


## 6. Temporalite


In [ ]:
monthly = (
    df.set_index('date_parsed')
    .resample('M')
    .agg(volume=('primary_type', 'size'), arrest_rate=('arrest', 'mean'))
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(monthly.index, monthly['volume'], color='#4c78a8', linewidth=2)
axes[0].set_title('Volume mensuel de crimes', fontweight='bold')
axes[0].set_ylabel('Volume')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].plot(monthly.index, monthly['arrest_rate'] * 100, color='#f28e2b', linewidth=2)
axes[1].set_title('Taux d arrestation mensuel', fontweight='bold')
axes[1].set_ylabel('Arrestation (%)')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(EDA_DIR / '03_monthly_trends.png', bbox_inches='tight')
plt.show()


In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hour_day = (
    df.pivot_table(index='day_name', columns='hour', values='primary_type', aggfunc='count')
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(hour_day, cmap='Blues', ax=ax, linewidths=0.1)
ax.set_title('Volume par jour et heure', fontweight='bold')
ax.set_xlabel('Heure')
ax.set_ylabel('Jour')
plt.tight_layout()
plt.savefig(EDA_DIR / '04_hour_day_heatmap.png', bbox_inches='tight')
plt.show()


## 7. Lieux et geographie


In [ ]:
location_stats = (
    df.groupby('location_group')
    .agg(volume=('location_group', 'size'), arrest_rate=('arrest', 'mean'), domestic_rate=('domestic', 'mean'))
    .sort_values('volume', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(location_stats.index, location_stats['volume'], color='#4c78a8', edgecolor='white')
axes[0].set_title('Volume par groupe de lieu', fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].bar(location_stats.index, location_stats['arrest_rate'] * 100, color='#59a14f', edgecolor='white')
axes[1].set_title('Taux d arrestation par groupe de lieu', fontweight='bold')
axes[1].set_ylabel('Arrestation (%)')
axes[1].tick_params(axis='x', rotation=35)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(EDA_DIR / '05_location_groups.png', bbox_inches='tight')
plt.show()

location_stats.style.format({'volume': '{:,.0f}', 'arrest_rate': '{:.1%}', 'domestic_rate': '{:.1%}'})


In [ ]:
geo_sample = df.sample(min(25_000, len(df)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(
    geo_sample['longitude'], geo_sample['latitude'],
    c=geo_sample['arrest'], cmap='coolwarm', s=3, alpha=0.35
)
ax.set_title('Echantillon geographique - couleur = arrestation', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.spines[['top', 'right']].set_visible(False)
plt.colorbar(scatter, ax=ax, label='Arrestation')
plt.tight_layout()
plt.savefig(EDA_DIR / '06_geo_scatter_arrest.png', bbox_inches='tight')
plt.show()


## 8. Synthese EDA


In [ ]:
insights = {
    'periode': f'{df["date_parsed"].min().date()} -> {df["date_parsed"].max().date()}',
    'crime_plus_frequent': df['primary_type'].value_counts().idxmax(),
    'taux_arrestation_global': f'{df["arrest"].mean():.1%}',
    'groupe_lieu_plus_frequent': df['location_group'].value_counts().idxmax(),
    'heure_pic_volume': int(df['hour'].value_counts().idxmax()),
    'jour_pic_volume': df['day_name'].value_counts().idxmax(),
}

pd.Series(insights).to_frame('insight')
